# Aplicações com YOLO

Um detector devolve caixas, e uma aplicação precisa de respostas como quantas vagas estão livres ou quantos carros passaram. Entre as duas fica a lógica de aplicação. Este notebook usa o YOLO26 pré-treinado no COCO em duas aplicações: a ocupação das vagas de um estacionamento em uma foto e a contagem dos veículos que cruzam uma linha em um vídeo, em cada sentido.

In [ ]:
import os
import subprocess
import urllib.request
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video
from torchvision.ops import box_iou
from ultralytics import YOLO

As caixas estão no formato `[x0, y0, x1, y1]`, com $(x_0, y_0)$ o canto superior esquerdo. A `save_video` grava frames em um MP4 que o navegador consegue tocar.

In [ ]:
detector = YOLO("yolo26n.pt")
VEHICLES = [2, 3, 5, 7]   # classes do COCO: car, motorcycle, bus, truck


def download(url, name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


def show(image, width=10):
    plt.figure(figsize=(width, width * image.shape[0] / image.shape[1]))
    plt.imshow(image[:, :, ::-1])   # OpenCV usa BGR, o matplotlib espera RGB
    plt.axis("off")
    plt.show()


def draw_box(image, box, color, text="", thickness=2):
    x0, y0, x1, y1 = map(int, box)
    cv2.rectangle(image, (x0, y0), (x1, y1), color, thickness)
    cv2.putText(image, text, (x0, y0 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *args], check=True)


def save_video(frames, path, fps=25):
    writer = cv2.VideoWriter("data/temp.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (1280, 720))
    for frame in frames:
        writer.write(frame)
    writer.release()
    ffmpeg("-i", "data/temp.mp4", "-c:v", "h264", "-pix_fmt", "yuv420p", path)   # o navegador só toca H.264
    return Video(path, embed=True, width=800)

## Estacionamento

A foto é do dataset PKLot. As 28 vagas foram marcadas uma vez, na instalação da câmera, e cada uma tem o gabarito de ocupação.

In [ ]:
# Uma caixa [x0, y0, x1, y1] por vaga, em pixels
spots = [
    [644, 549, 805, 698], [701, 451, 849, 586], [747, 399, 879, 499], [771, 323, 900, 430],
    [810, 282, 918, 368], [832, 231, 938, 309], [851, 194, 956, 265], [874, 166, 978, 226],
    [897, 137, 982, 189], [470, 563, 648, 685], [678, 281, 797, 348], [711, 242, 827, 299],
    [736, 198, 853, 254], [760, 168, 868, 213], [798, 142, 885, 179], [361, 250, 438, 327],
    [408, 213, 486, 280], [458, 179, 534, 246], [495, 150, 569, 204], [530, 120, 606, 176],
    [563, 99, 635, 150], [189, 275, 272, 364], [252, 241, 334, 314], [316, 203, 391, 272],
    [366, 170, 435, 228], [410, 140, 480, 201], [450, 118, 522, 175], [491, 98, 554, 148],
]
truth = [1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1]   # 1 é ocupada, 0 é livre

parking = cv2.imread(download("https://huggingface.co/datasets/Voxel51/PKLot/resolve/main/data/data_5/2012-12-12_13_05_08.jpg", "pklot.jpg"))
image = parking.copy()
for k, spot in enumerate(spots):
    draw_box(image, spot, (0, 220, 255), str(k))
show(image)

Uma vaga está ocupada se algum veículo tem IoU com ela acima de um limiar:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

A `box_iou` do torchvision calcula a IoU de cada vaga com cada veículo, em uma matriz $[28, N]$. Na figura, vagas livres aparecem em verde, ocupadas em vermelho, erros com borda grossa e detecções em azul.

In [ ]:
def check_parking(threshold=0.3, imgsz=640):
    # Caixas dos veículos na foto, tensor [N, 4] com [x0, y0, x1, y1] em cada linha
    boxes = detector.predict(parking, classes=VEHICLES, imgsz=imgsz, verbose=False)[0].boxes.xyxy.cpu()

    # IoU de cada vaga com cada veículo, matriz [28, N]. A vaga está ocupada se algum veículo passa do limiar
    # occupied é uma lista com um bool por vaga, ex.: [True, False, ...]
    occupied = (box_iou(torch.tensor(spots, dtype=torch.float), boxes) > threshold).any(dim=1).tolist()

    # Quantas vagas batem com o gabarito
    accuracy = np.mean([o == t for o, t in zip(occupied, truth)])

    # Detecções em azul, vagas ocupadas em vermelho, livres em verde e erros com borda grossa
    image = parking.copy()
    for box in boxes:
        draw_box(image, box, (230, 120, 0), thickness=1)
    for spot, o, t in zip(spots, occupied, truth):
        draw_box(image, spot, (0, 0, 230) if o else (0, 200, 0), thickness=2 if o == t else 6)
    print(f"livres: {occupied.count(False)}/{len(spots)}, acerto: {accuracy:.0%}")
    show(image)


check_parking()

Os carros do fundo não foram detectados: o YOLO reduz a imagem para 640 pixels e eles ficam pequenos demais. Com `imgsz=1280`, a resolução original da foto, todas as vagas são classificadas corretamente.

In [ ]:
check_parking(imgsz=1280)

## Contagem de tráfego

O vídeo mostra uma rodovia vista de frente. Reduzimos o vídeo para 1280×720.

In [ ]:
vehicles_4k = download("https://media.roboflow.com/supervision/video-examples/vehicles.mp4", "vehicles.mp4")
ffmpeg("-i", vehicles_4k, "-vf", "scale=1280:720", "-c:v", "h264", "-pix_fmt", "yuv420p", "data/vehicles-720p.mp4")
Video("data/vehicles-720p.mp4", embed=True, width=800)

Traçamos uma linha horizontal na altura `line_y` e contamos os veículos cujo centro da caixa muda de lado entre dois frames. Como o eixo $y$ cresce para baixo, quem desce na imagem vem em direção à câmera e quem sobe se afasta.

Para saber que o carro de um frame é o mesmo do frame anterior, usamos o rastreamento do YOLO: o `track` associa as detecções de frames consecutivos e dá a cada objeto um `id` que se mantém ao longo do vídeo. Guardamos o último $y$ de cada `id` em `last_y`.

In [ ]:
line_y = 450
counts = Counter()   # quantos cruzamentos de cada (classe, sentido), ex.: {("car", "desce"): 4}
last_y = {}          # último y do centro de cada id, ex.: {7: 431.5, 12: 280.0}


def count_traffic(path):
    tracker = YOLO("yolo26n.pt")   # modelo novo, porque o rastreador guarda o que viu
    for result in tracker.track(path, classes=VEHICLES, stream=True, verbose=False):
        frame = result.orig_img
        boxes = result.boxes
        if boxes.id is not None:   # no começo do vídeo pode ainda não ter ids
            # box é [x0, y0, x1, y1], id e cls são inteiros
            for box, id, cls in zip(boxes.xyxy.tolist(), boxes.id.int().tolist(), boxes.cls.int().tolist()):
                label = result.names[cls]
                y = (box[1] + box[3]) / 2   # y do centro da caixa

                # Cruzou a linha se o centro mudou de lado desde o último frame
                if id in last_y and last_y[id] < line_y <= y:
                    counts[(label, "desce")] += 1
                if id in last_y and y < line_y <= last_y[id]:
                    counts[(label, "sobe")] += 1
                last_y[id] = y

                draw_box(frame, box, (0, 200, 0), f"{label} {id}")

        # Linha e placar
        cv2.line(frame, (0, line_y), (1280, line_y), (0, 220, 255), 3)
        for k, ((label, direction), n) in enumerate(sorted(counts.items())):
            cv2.putText(frame, f"{label} {direction}: {n}", (10, 30 * (k + 1)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 255), 2)
        yield frame


video = save_video(count_traffic("data/vehicles-720p.mp4"), "data/vehicles-counted.mp4")
print(dict(counts))
video

Erros comuns de contadores assim: o mesmo veículo classificado ora como `car`, ora como `truck`; o rastreador trocar o `id` de um veículo bem em cima da linha, perdendo o cruzamento; e veículos encobertos que nunca são detectados.